# Campus Crowd Monitor — Exploration Notebook

This notebook walks through every preprocessing and feature-extraction step **interactively**,
visualising intermediate outputs so you can tune parameters and include figures in your report.

---
**Syllabus units covered:** Unit 1, 3, 4


In [ ]:
import sys, os
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from preprocessing import (
    resize_frame, to_grayscale, gaussian_blur, apply_clahe,
    apply_clahe_color, BackgroundSubtractor, build_gaussian_pyramid
)
from feature_extraction import (
    canny_edges, harris_corners, hough_lines, draw_hough_lines,
    HOGFeatureExtractor, extract_sift, draw_keypoints, dog_edges
)
from detector import HOGPersonDetector, count_per_zone, occupancy_label
from segmentation import (
    region_growing, watershed_segment, draw_watershed_overlay,
    meanshift_segment, draw_zone_overlay, DEFAULT_ZONES
)
from density_map import make_density_map, overlay_heatmap, density_to_heatmap

plt.rcParams['figure.dpi'] = 110
print('Imports OK')

## 1. Load a test frame
Point `VIDEO_PATH` at your sample footage.  If you don't have a video yet, we generate a synthetic crowd-like frame for demonstration.

In [ ]:
VIDEO_PATH = '../data/sample_videos/canteen_sample.mp4'

if os.path.exists(VIDEO_PATH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    # Grab a frame from the middle of the video
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, raw_frame = cap.read()
    cap.release()
    assert ret, 'Could not read frame'
    print(f'Loaded frame from video: {raw_frame.shape}')
else:
    # Synthetic frame for demo
    raw_frame = np.random.randint(80, 180, (720, 1280, 3), dtype=np.uint8)
    # Add some 'people'-shaped blobs
    for _ in range(8):
        x = np.random.randint(50, 1200)
        y = np.random.randint(50, 650)
        cv2.ellipse(raw_frame, (x, y), (20, 50), 0, 0, 360, (60,60,60), -1)
    print('No video found — using synthetic frame')

plt.figure(figsize=(10,5))
plt.imshow(cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB))
plt.title('Raw frame'); plt.axis('off'); plt.show()

## 2. Preprocessing pipeline
Visualise each step side-by-side.

In [ ]:
# Step-by-step preprocessing
resized   = resize_frame(raw_frame, (640, 480))
gray      = to_grayscale(resized)
blurred   = gaussian_blur(gray, (5,5), sigma=1.0)
equalized = apply_clahe(gray, clip_limit=2.0)
enhanced  = apply_clahe_color(resized)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Resized', 'Grayscale', 'Gaussian blur', 'CLAHE equalized']
images = [cv2.cvtColor(resized, cv2.COLOR_BGR2RGB), gray, blurred, equalized]
cmaps  = [None, 'gray', 'gray', 'gray']

for ax, img, title, cmap in zip(axes, images, titles, cmaps):
    ax.imshow(img, cmap=cmap); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.savefig('../report/fig_preprocessing.png', dpi=150); plt.show()
print('Saved: report/fig_preprocessing.png')

## 3. Histogram — before and after CLAHE
Show that CLAHE spreads the intensity distribution.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.hist(gray.ravel(), bins=256, range=(0,255), color='steelblue', density=True)
ax1.set_title('Intensity histogram — original'); ax1.set_xlabel('Pixel intensity')

ax2.hist(equalized.ravel(), bins=256, range=(0,255), color='coral', density=True)
ax2.set_title('Intensity histogram — after CLAHE'); ax2.set_xlabel('Pixel intensity')

plt.tight_layout(); plt.savefig('../report/fig_histogram.png', dpi=150); plt.show()

## 4. Edge detection — Canny, LOG, DOG

In [ ]:
from feature_extraction import log_edges, dog_edges

canny = canny_edges(blurred, low_thresh=50, high_thresh=150)
log   = log_edges(blurred, sigma=1.5)
dog   = dog_edges(blurred, sigma1=1.0, sigma2=2.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, img, title in zip(axes,
    [canny, log, dog],
    ['Canny (low=50, high=150)', 'LOG (σ=1.5)', 'DOG (σ1=1, σ2=2)']):
    ax.imshow(img, cmap='gray'); ax.set_title(title); ax.axis('off')

plt.tight_layout(); plt.savefig('../report/fig_edges.png', dpi=150); plt.show()

## 5. Harris corner detection

In [ ]:
corners_img, corner_map = harris_corners(blurred, block_size=2, ksize=3, k=0.04)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(cv2.cvtColor(corners_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Harris corners (red)'); axes[0].axis('off')
axes[1].imshow(corner_map, cmap='hot')
axes[1].set_title('Corner response map R'); axes[1].axis('off')
plt.tight_layout(); plt.savefig('../report/fig_harris.png', dpi=150); plt.show()

## 6. HOG person detection + density heatmap

In [ ]:
detector = HOGPersonDetector(scale=1.05, hit_threshold=0.0, nms_thresh=0.45)
enhanced_bgr = apply_clahe_color(resized)
boxes, weights = detector.detect(enhanced_bgr)

print(f'Detected {len(boxes)} person(s)')

det_frame = detector.draw_detections(enhanced_bgr, boxes, weights)

# Density map
h, w = resized.shape[:2]
density = make_density_map((h, w), boxes, sigma=35)
blended = overlay_heatmap(enhanced_bgr, density, alpha=0.55, vmax=0.3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(det_frame, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'HOG detections ({len(boxes)} people)'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
axes[1].set_title('Density heatmap overlay'); axes[1].axis('off')
plt.tight_layout(); plt.savefig('../report/fig_detection_heatmap.png', dpi=150); plt.show()

## 7. Image pyramid (scale-space)

In [ ]:
pyramid = build_gaussian_pyramid(blurred, levels=4)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, (ax, level) in enumerate(zip(axes, pyramid)):
    ax.imshow(level, cmap='gray')
    ax.set_title(f'Level {i}: {level.shape[1]}×{level.shape[0]}')
    ax.axis('off')
plt.suptitle('Gaussian pyramid (scale-space)', fontsize=13)
plt.tight_layout(); plt.savefig('../report/fig_pyramid.png', dpi=150); plt.show()

## 8. Zone overlay + occupancy labels

In [ ]:
zone_counts = count_per_zone(boxes, DEFAULT_ZONES)
print('Zone counts:', zone_counts)

zone_frame = draw_zone_overlay(enhanced_bgr, zone_counts,
                               thresholds=(5, 15))

plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(zone_frame, cv2.COLOR_BGR2RGB))
plt.title('Zone occupancy overlay'); plt.axis('off')
plt.tight_layout(); plt.savefig('../report/fig_zones.png', dpi=150); plt.show()

## 9. Evaluation — accuracy vs ground truth
Compare HOG detections against manually counted ground truth for 20 frames.

In [ ]:
# Placeholder: fill in gt_counts with your manual counts per frame
# gt_counts = [3, 5, 4, 7, ...]
# pred_counts = [run detector on each frame]

# Example evaluation once you have data:
# mae = np.mean(np.abs(np.array(pred_counts) - np.array(gt_counts)))
# print(f'Mean Absolute Error: {mae:.2f} persons')

print('Fill in gt_counts and pred_counts from your video annotations.')